<!-- @format -->

# Adult Income Predition - Deep Learning Pipeline

Xây dựng pipeline Deep Learning hoàn chỉnh cho bài toán phân loại thu nhập trên tập dữ liệu Adult Census.

**Pipeline gồm các bước:**

1. Tổng quan dữ liệu
2. Tiền xử lí dữ liệu
4. Huấn luyện mô hình
5. Đánh giá & so sánh tổng hợp


<!-- @format -->

## 1. Tổng quan dữ liệu

Phần này tóm tắt nhanh đặc trưng dữ liệu, phân phối target và các đặc trưng quan trọng nhất, giúp định hướng cho pipeline deep learning.


In [ ]:
# Import thư viện và load dữ liệu
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import modules.eda_dl as eda

import modules.deep_learning as dl

url = "https://raw.githubusercontent.com/Hanne2202/ml-group10-data/main/adult.csv"
df = eda.load_data(url)


In [ ]:
# Tổng quan nhanh dữ liệu
eda.dataset_overview(df)
num_cols, cat_cols = eda.get_column_types(df)


In [ ]:
# Kiểm tra missing value và giá trị bất thường
eda.check_missing_values(df)
eda.inspect_categorical_values(df, cat_cols)


<!-- @format -->

**Nhận xét:** Một số cột phân loại có giá trị '?' thay cho missing value. Cần chuẩn hóa về NaN trước khi xử lý tiếp.


In [ ]:
# Phân phối biến mục tiêu (income)
eda.plot_target_distribution(df, target_col='income')

<!-- @format -->

**Nhận xét:** Dữ liệu có mất cân bằng nhẹ giữa hai lớp thu nhập (>50K và <=50K), cần lưu ý khi đánh giá mô hình. Nhóm dự định sẽ sử dụng phương pháp SMOTE để cân bằng lại số lượng của 2 lớp này giúp tránh overfitting đối với người có income >50k.


In [ ]:
# Phân tích nhanh một số biến phân loại tiêu biểu
top_cat = ['education', 'occupation', 'marital-status']
eda.plot_categorical_by_target(df, top_cat, target_col='income')

<!-- @format -->

**Nhận xét:** Một số biến phân loại như education, occupation, marital-status có sự khác biệt rõ rệt giữa hai nhóm thu nhập.


In [ ]:
# Phân tích nhanh các biến số quan trọng
num_cols_sel = ['age', 'hours-per-week', 'educational-num']
eda.plot_numerical_by_target(df, num_cols_sel, target_col='income')


<!-- @format -->

**Nhận xét:** Các biến số như age, hours-per-week, educational-num đều có sự khác biệt phân phối giữa hai nhóm thu nhập.


In [ ]:
# Phân tích phân phối và ý nghĩa của capital-gain, capital-loss
eda.analyze_capital_feature(df, 'capital-gain', target_col='income')
eda.analyze_capital_feature(df, 'capital-loss', target_col='income')


<!-- @format -->

**Nhận xét:** Cả hai biến capital-gain và capital-loss đều có phân phối rất lệch, nhiều giá trị 0, nhưng tỷ lệ giá trị khác 0 ở nhóm >50K cao hơn đáng kể. Đây là đặc trưng hữu ích cho mô hình.


In [ ]:
# Heatmap tương quan các biến số
eda.plot_correlation_heatmap(df, num_cols)


<!-- @format -->

**Nhận xét:** Các biến số không có cặp nào tương quan quá cao, có thể giữ lại hầu hết cho mô hình.


In [ ]:
# Kiểm tra trùng lặp thông tin education & educational-num
eda.check_education_redundancy(df)

<!-- @format -->

**Nhận xét:** Hai cột education và educational-num gần như mang cùng thông tin, có thể cân nhắc chỉ giữ lại một trong hai khi xây dựng mô hình.


<!-- @format -->

# 2. Tiền xử lí dữ liệu


<!-- @format -->
## 2.1. Import preprocessing module

In [ ]:
import importlib
import modules.preprocessing_dl as prep_module
importlib.reload(prep_module)

from modules.preprocessing_dl import (
    drop_columns,
    drop_missing_values,
    map_target_variable,
    apply_ohe,
    scale_numeric,
    split_data,
    apply_smote,
)

<!-- @format -->
## 2.2. Làm sạch dữ liệu

Các bước:
1. **Xoá cột dư thừa**: `education` (trùng thông tin với `educational-num`) và `fnlwgt` (trọng số thống kê, không liên quan phân loại).
2. **Xử lý giá trị `?`**: thay bằng `NaN` rồi xoá các hàng thiếu.
3. **Mã hoá biến mục tiêu**: `<=50K` → 0, `>50K` → 1.

In [ ]:
df_clean = df.copy()

# 1. Xoá cột dư thừa
df_clean = drop_columns(df_clean, columns=['education', 'fnlwgt'])

# 2. Xử lý missing value ('?' → NaN → drop row)
df_clean = drop_missing_values(df_clean)

# 3. Mã hoá biến mục tiêu
df_clean = map_target_variable(df_clean, target_col='income')

print(f"\nKích thước sau làm sạch: {df_clean.shape}")
df_clean.head()

<!-- @format -->
## 2.3. Xác định loại đặc trưng

In [ ]:
TARGET_COL = 'income'

num_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

# Bỏ target khỏi danh sách (nếu có)
if TARGET_COL in num_cols: num_cols.remove(TARGET_COL)
if TARGET_COL in cat_cols: cat_cols.remove(TARGET_COL)

print(f"Numerical features ({len(num_cols)}): {num_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")

<!-- @format -->
## 2.4. Chia dữ liệu Train / Val / Test

Split **trước** khi encode & scale để tránh data leakage.
Tỉ lệ: **70% train – 10% val – 20% test** (stratify theo `income`).

In [ ]:
X_train_raw, X_val_raw, X_test_raw, y_train, y_val, y_test = split_data(
    df_clean,
    target_col=TARGET_COL,
    test_size=0.2,
    val_size=0.1,
    random_state=42,
)

<!-- @format -->
## 2.5. One-Hot Encoding (categorical)

Fit `OneHotEncoder` **chỉ trên train**, sau đó transform val/test bằng cùng encoder đó.

In [ ]:
X_train_ohe, ohe_encoder = apply_ohe(X_train_raw, cat_cols)
X_val_ohe, _= apply_ohe(X_val_raw,   cat_cols, encoder=ohe_encoder)
X_test_ohe, _= apply_ohe(X_test_raw,  cat_cols, encoder=ohe_encoder)

print(f"\nShape sau OHE  →  train: {X_train_ohe.shape} | val: {X_val_ohe.shape} | test: {X_test_ohe.shape}")

<!-- @format -->
## 2.6. Chuẩn hoá đặc trưng số (StandardScaler)

Fit `StandardScaler` **chỉ trên train**, transform val/test.

In [ ]:
X_train_sc, scaler = scale_numeric(X_train_ohe, num_cols)
X_val_sc,   _      = scale_numeric(X_val_ohe,   num_cols, scaler=scaler)
X_test_sc,  _      = scale_numeric(X_test_ohe,  num_cols, scaler=scaler)

<!-- @format -->
## 2.7. Cân bằng lớp bằng SMOTE

Dữ liệu có **mất cân bằng ~24%** ở lớp `>50K`.
SMOTE sinh các mẫu tổng hợp cho lớp thiểu số **chỉ trên tập train** để tránh data leakage.

> **Lưu ý**: SMOTE phải được gọi **sau khi encode & scale** (vì cần tính khoảng cách Euclidean trên không gian số).
> Val/test **không được** áp dụng SMOTE – phải giữ nguyên phân phối thực tế.

In [ ]:
X_train_sm, y_train_sm = apply_smote(
    X_train_sc,
    y_train,
    sampling_strategy='auto',   # cân bằng 50/50
    k_neighbors=5,
    random_state=42,
)

<!-- @format -->
## 2.8. Tóm tắt kết quả preprocessing

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    'Split':      ['Train (sau SMOTE)', 'Val (giữ nguyên)', 'Test (giữ nguyên)'],
    'Samples':    [len(X_train_sm), len(X_val_sc), len(X_test_sc)],
    'Features':   [X_train_sm.shape[1], X_val_sc.shape[1], X_test_sc.shape[1]],
    'Class >50K': [
        f"{y_train_sm.mean():.1%}",
        f"{y_val.mean():.1%}",
        f"{y_test.mean():.1%}",
    ],
})
print(summary.to_string(index=False))
print(f"\nEncoding artifacts saved:")
print(f"  ohe_encoder : {type(ohe_encoder).__name__}")
print(f"  scaler      : {type(scaler).__name__}")

<!-- @format -->
# 3. Logistic Regression (Baseline)

Logistic Regression được dùng làm **baseline** để so sánh với MLP sau này.

**Lý do chọn làm baseline:**
- Đơn giản, huấn luyện nhanh, dễ giải thích
- Hiệu quả tốt với dữ liệu đã được scale và OHE
- Kết quả của nó là ngưỡng tối thiểu mà MLP cần vượt qua

<!-- @format -->
## 3.1. Huấn luyện Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score,
)
import matplotlib.pyplot as plt
import numpy as np

# Dùng X_train_sm (sau SMOTE) để train
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    solver='lbfgs',
)
lr_model.fit(X_train_sm, y_train_sm)
print("Logistic Regression trained.")

<!-- @format -->
## 3.2. Đánh giá trên Val set

In [ ]:
y_val_pred_lr  = lr_model.predict(X_val_sc)
y_val_proba_lr = lr_model.predict_proba(X_val_sc)[:, 1]

print("=== Logistic Regression – Val Set ===")
print(classification_report(y_val, y_val_pred_lr, target_names=['<=50K', '>50K']))

val_metrics_lr = {
    'accuracy' : round(accuracy_score(y_val,  y_val_pred_lr),  4),
    'precision': round(precision_score(y_val, y_val_pred_lr),  4),
    'recall'   : round(recall_score(y_val,    y_val_pred_lr),  4),
    'f1'       : round(f1_score(y_val,        y_val_pred_lr),  4),
    'roc_auc'  : round(roc_auc_score(y_val,   y_val_proba_lr), 4),
}
print("Val metrics:", val_metrics_lr)

<!-- @format -->
## 3.3. Đánh giá trên Test set

In [ ]:
y_test_pred_lr  = lr_model.predict(X_test_sc)
y_test_proba_lr = lr_model.predict_proba(X_test_sc)[:, 1]

print("=== Logistic Regression – Test Set ===")
print(classification_report(y_test, y_test_pred_lr, target_names=['<=50K', '>50K']))

test_metrics_lr = {
    'accuracy' : round(accuracy_score(y_test,  y_test_pred_lr),  4),
    'precision': round(precision_score(y_test, y_test_pred_lr),  4),
    'recall'   : round(recall_score(y_test,    y_test_pred_lr),  4),
    'f1'       : round(f1_score(y_test,        y_test_pred_lr),  4),
    'roc_auc'  : round(roc_auc_score(y_test,   y_test_proba_lr), 4),
}
print("Test metrics:", test_metrics_lr)

# Confusion matrix
cm_lr = confusion_matrix(y_test, y_test_pred_lr)
disp  = ConfusionMatrixDisplay(cm_lr, display_labels=['<=50K', '>50K'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix – Logistic Regression')
plt.tight_layout()
plt.show()

<!-- @format -->
# 4. MLP – Multi-Layer Perceptron

So sánh MLP với Logistic Regression baseline.

**Kiến trúc MLP:**
- Input layer → Hidden layers (BatchNorm → ReLU → Dropout) → Output (1 logit)
- Loss: `BCEWithLogitsLoss` với `pos_weight` để xử lý class imbalance còn lại trên val/test
- Optimizer: Adam + ReduceLROnPlateau scheduler
- Early stopping theo `val_loss`

<!-- @format -->
## 4.1. Setup device & DataLoaders

In [ ]:
import importlib
import modules.deep_learning as dl
importlib.reload(dl)

import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Đóng gói dữ liệu đã preprocessing vào dict chuẩn của dl module
prep = {
    'X_train': X_train_sm.values.astype('float32'),
    'X_val'  : X_val_sc.values.astype('float32'),
    'X_test' : X_test_sc.values.astype('float32'),
    'y_train': y_train_sm.values.astype('int64'),
    'y_val'  : y_val.values.astype('int64'),
    'y_test' : y_test.values.astype('int64'),
    'input_dim': X_train_sm.shape[1],
}

train_loader, val_loader, test_loader = dl.create_dataloaders(prep, batch_size=256)

<!-- @format -->
## 4.2. MLP cơ bản [256, 128, 64]

In [ ]:
model_mlp = dl.MLP(
    input_dim=prep['input_dim'],
    hidden_dims=[256, 128, 64],
    dropout=0.4,
)
print(model_mlp)
print(f"Tổng số tham số: {sum(p.numel() for p in model_mlp.parameters()):,}")

In [ ]:
history_mlp = dl.train_model(
    model_mlp,
    train_loader,
    val_loader,
    epochs=100,
    lr=1e-3,
    weight_decay=1e-3,
    patience=20,
    device=device,
    use_pos_weight=False,
)

<!-- @format -->
## 4.3. Learning Curves

In [ ]:
dl.plot_learning_curves(history_mlp)

<!-- @format -->
## 4.4. Đánh giá MLP trên Test set

In [ ]:
test_metrics_mlp, y_true_mlp, y_pred_mlp, y_proba_mlp = dl.evaluate_model(
    model_mlp, test_loader, device=device
)

<!-- @format -->
## 4.5. So sánh Logistic Regression vs MLP

In [ ]:
import pandas as pd

compare = pd.DataFrame([
    {
        'Model'    : 'Logistic Regression (baseline)',
        'Accuracy' : test_metrics_lr['accuracy'],
        'Precision': test_metrics_lr['precision'],
        'Recall'   : test_metrics_lr['recall'],
        'F1-Score' : test_metrics_lr['f1'],
        'ROC-AUC'  : test_metrics_lr['roc_auc'],
    },
    {
        'Model'    : 'MLP [256, 128, 64]',
        'Accuracy' : test_metrics_mlp['accuracy'],
        'Precision': test_metrics_mlp['precision'],
        'Recall'   : test_metrics_mlp['recall'],
        'F1-Score' : test_metrics_mlp['f1_score'],
        'ROC-AUC'  : round(roc_auc_score(y_true_mlp, y_proba_mlp), 4),
    },
])

compare = compare.set_index('Model')
print(compare.to_string())
compare.style.highlight_max(axis=0, color='lightgreen')

In [ ]:
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, compare.loc['Logistic Regression (baseline)', metrics_to_plot],
               width, label='Logistic Regression', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, compare.loc['MLP [256, 128, 64]', metrics_to_plot],
               width, label='MLP', color='coral', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('Score')
ax.set_title('Logistic Regression vs MLP – Test Set Metrics')
ax.legend()
ax.bar_label(bars1, fmt='%.3f', padding=3, fontsize=8)
ax.bar_label(bars2, fmt='%.3f', padding=3, fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 5. Tối ưu hoá Siêu tham số với Optuna (Automatic Hyperparameter Tuning)

Tiếp theo, ta sử dụng thư viện **Optuna** đã được đóng gói trong file `modules/tuning_mlp.py` để thực hiện quá trình Auto-ML. Công cụ này sẽ tự động chạy thử nhiều cấu hình tham số khác nhau (số lớp, số node, lr, dropout, batch_size...) và tìm ra tập hợp trả về điểm **F1-Score** trên tập Validation cao nhất.

In [ ]:
import importlib
import modules.tuning_mlp as tune
importlib.reload(tune)

print("Bắt đầu tìm kiếm tham số tối ưu bằng Optuna...")
# Ở đây ta dùng bộ dữ liệu KHÔNG có SMOTE nhưng đã chuẩn hoá
prep_tuned = {
    'X_train': X_train_sc.values.astype('float32'),
    'X_val'  : X_val_sc.values.astype('float32'),
    'X_test' : X_test_sc.values.astype('float32'),
    'y_train': y_train.values.astype('int64'),
    'y_val'  : y_val.values.astype('int64'),
    'y_test' : y_test.values.astype('int64'),
    'input_dim': X_train_sc.shape[1],
}
train_loader_tuned, val_loader_tuned, test_loader_tuned = dl.create_dataloaders(prep_tuned, batch_size=256)

def get_mock_study_mlp():
    class MockStudy:
        def __init__(self, best_params):
            self.best_params = best_params
            self.best_value = 0.7049062049062049
    return MockStudy({
        "n_layers": 3,
        "hidden_dim_0": 320,
        "hidden_dim_1": 448,
        "hidden_dim_2": 32,
        "dropout": 0.35,
        "lr": 0.00015193253929678293,
        "weight_decay": 3.0308702059789768e-05,
        "batch_size": 512,
        "optimizer": "AdamW"
    })
study_mlp = get_mock_study_mlp() # Dùng mock study để tiết kiệm thời gian

### QUÁ TRÌNH NÀY TỐN TRUNG BÌNH 7-10 PHÚT NÊN ĐƯỢC COMMENT LẠI VÀ CHẠY BẰNG DỮ LIỆU HARDCODE TỪ CÁC LẦN CHẠY TRÊN MÁY SINH VIÊN ĐỂ TIẾT KIỆM THỜI GIAN, BỎ COMMENT PHÍA DƯỚI ĐỂ XEM ĐƯỢC QUÁ TRÌNH CHẠY

In [ ]:
# study_mlp = tune.run_optuna_search(
#     prep=prep_tuned, 
#     n_trials=30,  # Có thể tăng số lượng trial lên 50-100 nếu có nhiều thời gian (Tạm để 30 để chạy tốc hành)
#     epochs=50, 
#     patience=10, 
#     device=device
# )

# # Trực quan hoá lịch sử tìm kiếm và độ quan trọng của các tham số
# tune.plot_optuna_results(study_mlp)

In [ ]:
# 5.1. Huấn luyện lại và đánh giá mô hình bằng tham số đỉnh nhất (Best params)
best_model_optuna, best_history_optuna, best_metrics_optuna = tune.train_best_optuna_model(
    study=study_mlp,
    prep=prep_tuned,
    epochs=100,
    patience=15,
    device=device
)

# Vẽ lại learning curve của best model
dl.plot_learning_curves(best_history_optuna)

## 5.2. So sánh Logistic Regression vs MLP Optuna Best

In [ ]:
# Lấy dự đoán từ mô hình tốt nhất của Optuna để tính ROC-AUC
_, y_true_optuna, _, y_proba_optuna = dl.evaluate_model(best_model_optuna, test_loader_tuned, device=device)

compare_optuna = pd.DataFrame([
    {
        'Model'    : 'Logistic Regression (baseline)',
        'Accuracy' : test_metrics_lr['accuracy'],
        'Precision': test_metrics_lr['precision'],
        'Recall'   : test_metrics_lr['recall'],
        'F1-Score' : test_metrics_lr['f1'],
        'ROC-AUC'  : test_metrics_lr['roc_auc'],
    },
    {
        'Model'    : 'MLP (Optuna Best)',
        'Accuracy' : best_metrics_optuna['accuracy'],
        'Precision': best_metrics_optuna['precision'],
        'Recall'   : best_metrics_optuna['recall'],
        'F1-Score' : best_metrics_optuna['f1_score'],
        'ROC-AUC'  : round(roc_auc_score(y_true_optuna, y_proba_optuna), 4),
    },
])

compare_optuna = compare_optuna.set_index('Model')
display(compare_optuna.style.highlight_max(axis=0, color='lightgreen'))

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, compare_optuna.loc['Logistic Regression (baseline)', metrics_to_plot],
               width, label='Logistic Regression', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, compare_optuna.loc['MLP (Optuna Best)', metrics_to_plot],
               width, label='MLP (Optuna Best)', color='coral', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('Score')
ax.set_title('Logistic Regression vs MLP Optuna Best – Test Set Metrics')
ax.legend()
ax.bar_label(bars1, fmt='%.3f', padding=3, fontsize=8)
ax.bar_label(bars2, fmt='%.3f', padding=3, fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 6. TabNet Model (Attentive Interpretable Tabular Learning)

Sử dụng kiến trúc TabNet được tối ưu hóa đặc biệt cho dữ liệu dạng bảng.
Nhóm quyết định **KHÔNG sử dụng SMOTE** cho TabNet vì mô hình này yêu cầu giữ nguyên tính chất của các biến phân loại (để đưa qua lớp embedding layer) thay vì ép sang dạng One-Hot liên tục sinh bởi SMOTE.

In [ ]:
# 1. Gọi hàm tiền xử lý dữ liệu tự động cho TabNet từ module có sẵn
import importlib
import modules.deep_learning as dl
importlib.reload(dl) # Reload lại module để nhận code mới nhất từ file .py

import numpy as np

print("--- Chuẩn bị Data gốc cho TabNet ---")
# Đầu vào là df_clean (dataset chuẩn bị ở bước 2.2, chưa qua encode hay smote)
prep_tabnet = dl.preprocess_for_tabnet(
    df_clean, 
    target_col='income', 
    random_state=42
)

print("\nShape của dữ liệu train TabNet (số + phân loại đã mã hóa số nguyên):", prep_tabnet['X_train'].shape)

In [ ]:
# 2. Đào tạo mô hình TabNet

print("\n--- Bắt đầu Training TabNet ---")
tabnet_model = dl.train_tabnet_model(
    prep=prep_tabnet,
    n_d=32,               # Kích thước lớp quyết định
    n_a=32,               # Kích thước lớp chú ý
    n_steps=3,            # Số chặng Attention
    gamma=1.3,
    lr=2e-2,
    max_epochs=100,       
    patience=15,          # Dừng sớm nếu sau 15 epoch val loss không giảm
    batch_size=512,
    virtual_batch_size=128,
    device=device
)

In [ ]:
# 3. Đánh giá và So sánh lại với các Baseline trước đó
metrics_tabnet, y_true_tabnet, y_pred_tabnet, y_proba_tabnet = dl.evaluate_tabnet(tabnet_model, prep_tabnet)

compare_all = pd.DataFrame([
    {
        'Model'    : 'Logistic Regression (baseline)',
        'Accuracy' : test_metrics_lr['accuracy'],
        'Precision': test_metrics_lr['precision'],
        'Recall'   : test_metrics_lr['recall'],
        'F1-Score' : test_metrics_lr['f1'],
        'ROC-AUC'  : test_metrics_lr['roc_auc'],
    },
    {
        'Model'    : 'MLP (Optuna Best)',
        'Accuracy' : best_metrics_optuna['accuracy'],
        'Precision': best_metrics_optuna['precision'],
        'Recall'   : best_metrics_optuna['recall'],
        'F1-Score' : best_metrics_optuna['f1_score'],
        'ROC-AUC'  : round(roc_auc_score(y_true_optuna, y_proba_optuna), 4),
    },
    {
        'Model'    : 'TabNet Basic (No SMOTE)',
        'Accuracy' : metrics_tabnet['accuracy'],
        'Precision': metrics_tabnet['precision'],
        'Recall'   : metrics_tabnet['recall'],
        'F1-Score' : metrics_tabnet['f1_score'],
        'ROC-AUC'  : round(roc_auc_score(y_true_tabnet, y_proba_tabnet), 4),
    },
])

compare_all = compare_all.set_index('Model')
display(compare_all.style.highlight_max(axis=0, color='lightgreen'))

# Trực quan hoá
fig, ax = plt.subplots(figsize=(12, 6))
width_new = 0.25
for i, model_name in enumerate(compare_all.index):
    ax.bar(x + (i - 1) * width_new, compare_all.loc[model_name, metrics_to_plot],
           width_new, label=model_name, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('Score')
ax.set_title('Đại chiến Model – Test Set Metrics')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 7. Tối ưu hoá Siêu tham số TabNet với Optuna

Chúng ta sẽ sử dụng các hàm đã được gói sẵn trong `modules/tuning_mlp.py` để tìm kiếm siêu tham số và huấn luyện mô hình TabNet tốt nhất. Đặc biệt ở đây còn kèm theo **Cơ chế Pretraining (Self-supervised learning)** tích hợp sẵn.

In [ ]:
import modules.tuning_mlp as tune
import importlib
importlib.reload(tune)

# 7.1 Thực hiện dò TabNet params
# Sử dụng hàm dò Optuna có sẵn của bạn trong tuning_mlp.py (Đã bật use_pretraining=True mặc định)
def get_mock_study_tabnet(*args, **kwargs):
        class MockStudy:
            def __init__(self, best_params):
                self.best_params = best_params
                self.best_value = 0.6885694729637235
        return MockStudy({
            "width": 8,
            "n_steps": 4,
            "gamma": 1.0137868611216592,
            "lambda_sparse": 0.003924034647132935,
            "lr": 0.007085132090225794,
            "batch_size": 512,
            "virtual_batch_size": 256,
            "pretrain_epochs": 50,
            "pretraining_ratio": 0.6195014675689636
        })
study_tabnet = get_mock_study_tabnet() # Dùng mock study để tiết kiệm thời gian

### QUÁ TRÌNH NÀY TỐN TRUNG BÌNH 30-40 PHÚT NÊN ĐƯỢC COMMENT LẠI VÀ CHẠY BẰNG DỮ LIỆU HARDCODE TỪ CÁC LẦN CHẠY TRÊN MÁY SINH VIÊN ĐỂ TIẾT KIỆM THỜI GIAN, BỎ COMMENT PHÍA DƯỚI ĐỂ XEM ĐƯỢC QUÁ TRÌNH CHẠY

In [ ]:
# study_tabnet = tune.run_optuna_search_tabnet(
#     prep_tabnet=prep_tabnet,
#     n_trials=10, 
#     max_epochs=50,
#     patience=10,
#     device=device,
#     use_pretraining=True # Sử dụng pretrain tích hợp sẵn
# )

In [ ]:
# 7.2 Lấy mô hình có cấu hình tốt nhất và trong tuning_mlp.py
# Hàm `train_best_optuna_tabnet_model` sẽ trả về (model, metrics) nên ta lấy cả hai
best_tabnet_optuna, metrics_tabnet_best = tune.train_best_optuna_tabnet_model(
    study=study_tabnet, 
    prep_tabnet=prep_tabnet, 
    max_epochs=100, 
    patience=15, 
    device=device,
    use_pretraining=True
)

# Để lấy phân loại ROC-AUC, ta dùng predict_proba hoặc chạy lại evaluate
_, y_true_tb, y_pred_tb, y_proba_tb = dl.evaluate_tabnet(best_tabnet_optuna, prep_tabnet)

compare_all.loc['TabNet (Optuna Best)'] = [
    metrics_tabnet_best['accuracy'],
    metrics_tabnet_best['precision'],
    metrics_tabnet_best['recall'],
    metrics_tabnet_best['f1_score'],
    round(roc_auc_score(y_true_tb, y_proba_tb), 4)
]

display(compare_all.style.highlight_max(axis=0, color='lightgreen'))

# Trực quan hoá bản chốt hạ cuối
fig, ax = plt.subplots(figsize=(14, 7))
width_new = 0.2
for i, model_name in enumerate(compare_all.index):
    # Dịch chuyển vị trí các cột
    ax.bar(x + (i - 1.5) * width_new, compare_all.loc[model_name, metrics_to_plot],
           width_new, label=model_name, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('Score')
ax.set_title('Đại chiến Chung cuộc – Test Set Metrics')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()